# 🎈 Part 1: การตรวจจับภาพซ้ำ (Duplicate Detection) และ Multi-Label Stratified Splitting (80:10:10)
### โครงการ: Red Balloon Interceptor (ตรวจจับและสกัดกั้นลูกโป่งสีแดงด้วย Raspberry Pi 4)

---

### 🎯 วัตถุประสงค์ของ Notebook นี้:
1. **วิเคราะห์ Dataset Pascal VOC (`baloon.v2i.voc`)**: ตรวจสอบจำนวนภาพ, Bounding Box, คลาส (`red`, `blue`, `green`), และจำนวน Object ต่อภาพ
2. **ตรวจจับภาพซ้ำ (Duplicate & Near-Duplicate Detection)**:
   - **Exact Duplicate**: ตรวจสอบความเหมือนระดับไบต์ 100% ด้วย Cryptographic Hash (**MD5 / SHA-256**)
   - **Perceptual Near-Duplicate**: ตรวจสอบภาพถ่ายรัว (Burst shots) หรือภาพที่มีความคล้ายคลึงกันสูงด้วย **Perceptual Hashing (dHash / pHash / aHash)** พร้อมคำนวณ Hamming Distance และแสดงผลเปรียบเทียบแบบ Side-by-Side
3. **ทำ Multi-Label & Multi-Aspect Stratified Splitting (80:10:10)**:
   - คุมสัดส่วน **Train: 80% (280 ภาพ) | Val: 10% (35 ภาพ) | Test: 10% (35 ภาพ)**
   - คุมสัดส่วนของ **ทุกคลาส (`red`, `blue`, `green`)** ให้กระจายตัวเท่ากัน ~80:10:10
   - คุมการกระจายตัวของ **จำนวน Object ต่อรูป (0, 1, 2, 3, 4 ลูก)** ให้เท่ากันทุก Split
4. **Export Dataset โครงสร้างมาตรฐาน Pascal VOC**:
   - บันทึกไฟล์ลงโฟลเดอร์ `balloon_dataset_stratified/` (`train/`, `val/`, `test/`)
   - สร้างไฟล์สรุป Metadata `split_manifest.csv` และ `dataset_summary.json` สำหรับส่งต่อไปเทรนใน **Notebook 2**


In [ ]:
# ติดตั้ง Library ที่จำเป็น (สำหรับ Google Colab หรือ Environment ใหม่)
# !pip install numpy pandas matplotlib seaborn Pillow imagehash tqdm

import os
import glob
import shutil
import hashlib
import json
import csv
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
import imagehash
from tqdm import tqdm

# กำหนด Random Seed เพื่อให้ผลการสุ่มซ้ำได้แน่นอน (Reproducible)
SEED = 42
np.random.seed(SEED)

# ระบบตรวจจับ Path อัตโนมัติ (รองรับการเปิดจาก root หรือข้างใน ComputerVisionSRC)
if os.path.exists("baloon.v2i.voc"):
    RAW_DATASET_DIR = "baloon.v2i.voc"
    OUTPUT_DATASET_DIR = "balloon_dataset_stratified"
elif os.path.exists("ComputerVisionSRC/baloon.v2i.voc"):
    RAW_DATASET_DIR = "ComputerVisionSRC/baloon.v2i.voc"
    OUTPUT_DATASET_DIR = "ComputerVisionSRC/balloon_dataset_stratified"
else:
    RAW_DATASET_DIR = "/Users/papajhon/Project/Red Balloon Intercepter/ComputerVisionSRC/baloon.v2i.voc"
    OUTPUT_DATASET_DIR = "/Users/papajhon/Project/Red Balloon Intercepter/ComputerVisionSRC/balloon_dataset_stratified"

print("✅ Library Loaded Successfully!")
print(f"📁 Raw Dataset Directory: {os.path.abspath(RAW_DATASET_DIR)}")
print(f"📁 Output Split Directory: {os.path.abspath(OUTPUT_DATASET_DIR)}")


## 1. การอ่านและพาร์สข้อมูล Pascal VOC XML Annotations
เราจะเขียนฟังก์ชัน `parse_voc_xml` เพื่อดึงข้อมูล:
- ขนาดภาพ (Width, Height)
- รายการ Bounding Box `[xmin, ymin, xmax, ymax]`
- ป้ายชื่อคลาส (`red`, `blue`, `green`)
- นับจำนวน Object แต่ละสีในแต่ละภาพ


In [ ]:
def parse_voc_xml(xml_path):
    """
    อ่านไฟล์ XML มาตรฐาน Pascal VOC และสกัดข้อมูล Bounding Boxes, คลาส, และขนาดภาพ
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    size_elem = root.find("size")
    width = int(size_elem.find("width").text) if size_elem is not None else 640
    height = int(size_elem.find("height").text) if size_elem is not None else 640
    
    objects = []
    classes = []
    
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        classes.append(name)
        
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        
        # ป้องกันกรณี Bounding Box ผิดปกติ
        xmin = max(0.0, min(float(width), xmin))
        ymin = max(0.0, min(float(height), ymin))
        xmax = max(0.0, min(float(width), xmax))
        ymax = max(0.0, min(float(height), ymax))
        
        objects.append({
            "name": name,
            "box": [xmin, ymin, xmax, ymax]
        })
        
    return {
        "width": width,
        "height": height,
        "num_objects": len(objects),
        "classes": classes,
        "objects": objects
    }

# ค้นหาไฟล์ XML ทั้งหมดในชุดข้อมูลเดิม (รวม train, valid, test)
raw_xml_files = sorted(glob.glob(os.path.join(RAW_DATASET_DIR, "*", "*.xml")))
print(f"🔍 ตรวจพบไฟล์ Annotation ทั้งหมด: {len(raw_xml_files)} ไฟล์")

records = []
for xml_p in tqdm(raw_xml_files, desc="Parsing VOC XMLs"):
    img_p = xml_p.replace(".xml", ".jpg")
    if not os.path.exists(img_p):
        continue
        
    info = parse_voc_xml(xml_p)
    counts = Counter(info["classes"])
    
    records.append({
        "filename": os.path.basename(img_p),
        "img_path": img_p,
        "xml_path": xml_p,
        "width": info["width"],
        "height": info["height"],
        "num_objects": info["num_objects"],
        "red": counts.get("red", 0),
        "blue": counts.get("blue", 0),
        "green": counts.get("green", 0),
        "objects": info["objects"]
    })

df = pd.DataFrame(records)
print(f"✅ โหลดข้อมูลลง DataFrame สำเร็จ: {len(df)} รายการ")
df[["filename", "num_objects", "red", "blue", "green"]].head(10)


## 2. การวิเคราะห์สถิติและการกระจายตัวของชุดข้อมูล (Exploratory Data Analysis)
ตรวจสอบภาพรวม:
- จำนวนภาพทั้งหมด: **350 ภาพ**
- จำนวน Bounding Box ทั้งหมด: **390 กล่อง**
- การกระจายตัวของจำนวนลูกโป่งในภาพ (0 ถึง 4 ลูก)
- สัดส่วนของแต่ละสี (`red` = 156, `blue` = 117, `green` = 117)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. กราฟแจกแจงจำนวน Object ต่อภาพ
obj_dist = df["num_objects"].value_counts().sort_index()
sns.barplot(x=obj_dist.index, y=obj_dist.values, ax=axes[0], palette="Blues_d", hue=obj_dist.index, legend=False)
axes[0].set_title("Distribution of Objects per Image", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Number of Balloons", fontsize=12)
axes[0].set_ylabel("Number of Images", fontsize=12)
for i, v in enumerate(obj_dist.values):
    axes[0].text(i, v + 4, str(v), ha="center", fontweight="bold", fontsize=11)

# 2. กราฟจำนวน Bounding Box แยกตามคลาส
class_counts = pd.Series({
    "Red (Target)": df["red"].sum(),
    "Blue": df["blue"].sum(),
    "Green": df["green"].sum()
})
colors = ["#e74c3c", "#3498db", "#2ecc71"]
axes[1].bar(class_counts.index, class_counts.values, color=colors, edgecolor="black", alpha=0.85)
axes[1].set_title("Total Bounding Boxes by Class", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Total Instances", fontsize=12)
for i, v in enumerate(class_counts.values):
    axes[1].text(i, v + 3, str(v), ha="center", fontweight="bold", fontsize=11)

plt.tight_layout()
plt.show()

print(f"📊 สรุปข้อมูลสถิติภาพรวม:")
print(f"• จำนวนภาพทั้งหมด: {len(df)} ภาพ (ขนาด {df['width'].iloc[0]}x{df['height'].iloc[0]})")
print(f"• จำนวน Bounding Box ทั้งหมด: {df['num_objects'].sum()} กล่อง")
print(f"• คลาส Red (เป้าหมายหลัก): {df['red'].sum()} กล่อง")
print(f"• คลาส Blue: {df['blue'].sum()} กล่อง")
print(f"• คลาส Green: {df['green'].sum()} กล่อง")
print(f"• ภาพ Background (0 Object): {(df['num_objects'] == 0).sum()} ภาพ")


## 3. การตรวจจับภาพซ้ำ (Duplicate & Near-Duplicate Detection)
การตรวจจับภาพซ้ำแบ่งเป็น 2 มิติ:
1. **Exact Duplicate (ภาพซ้ำตรงกันทุกไบต์ 100%)**:
   - ใช้ **MD5 / SHA-256** แฮชไบนารีของไฟล์
2. **Perceptual Hash (Near-Duplicate / ภาพที่ถ่ายรัวหรือมุมซ้ำกันมาก)**:
   - **dHash (Difference Hash)**: เปรียบเทียบ Gradient แนวนอนของระดับสีเทา ทนต่อแสงและ Contrast
   - **pHash (Perceptual Hash)**: ใช้ Discrete Cosine Transform (DCT) ดึงรูปทรงหลัก ทนต่อความเบลอและการหมุน
   - **Hamming Distance**: วัดระยะห่างของ Hash หาก $\le 4$ ถือว่าภาพคล้ายกันมากเป็นคู่แข่งภาพซ้ำ


In [ ]:
# ฟังก์ชันคำนวณ MD5
def get_file_md5(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# ฟังก์ชันคำนวณ Perceptual Hashes
def get_perceptual_hashes(img_path):
    with Image.open(img_path) as img:
        img_rgb = img.convert("RGB")
        dh = imagehash.dhash(img_rgb)
        ph = imagehash.phash(img_rgb)
        ah = imagehash.average_hash(img_rgb)
    return str(dh), str(ph), str(ah)

print("กำลังคำนวณ Hashes สำหรับภาพทั้งหมด 350 ภาพ...")
md5s, dhashes, phashes, ahashes = [], [], [], []

for p in tqdm(df["img_path"], desc="Hashing images"):
    md5s.append(get_file_md5(p))
    dh, ph, ah = get_perceptual_hashes(p)
    dhashes.append(dh)
    phashes.append(ph)
    ahashes.append(ah)

df["md5"] = md5s
df["dhash"] = dhashes
df["phash"] = phashes
df["ahash"] = ahashes

# ตรวจสอบ Exact Duplicate
exact_duplicates = df[df.duplicated("md5", keep=False)]
print(f"\n🔍 ผลการตรวจ Exact Duplicate (MD5): พบ {len(exact_duplicates)} ภาพ")
if len(exact_duplicates) > 0:
    print(exact_duplicates[["filename", "md5"]])
else:
    print("✅ ไม่มีภาพที่ซ้ำกันแบบตรงกัน 100% ทุกไฟล์มี Binary Signature ต่างกัน")


### ตรวจสอบ Near-Duplicate Images ด้วย Pairwise Hamming Distance
เราจะค้นหาภาพคู่ที่มี Hamming Distance ต่ำกว่าหรือเท่ากับ Threshold ($\le 4$) เพื่อตรวจสอบว่ามีภาพที่เป็น Burst shots หรือมุมกล้องซ้ำกันหรือไม่


In [ ]:
def find_duplicate_pairs(dataframe, hash_col="dhash", threshold=4):
    """
    คำนวณ Hamming Distance ระหว่างคู่ภาพทั้งหมดเพื่อหาภาพที่มีความคล้ายคลึงกันสูง
    """
    hashes = [imagehash.hex_to_hash(h) for h in dataframe[hash_col]]
    n = len(hashes)
    pairs = []
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = hashes[i] - hashes[j]
            if dist <= threshold:
                pairs.append({
                    "idx1": i,
                    "idx2": j,
                    "file1": dataframe["filename"].iloc[i],
                    "file2": dataframe["filename"].iloc[j],
                    "distance": dist,
                    "img1_path": dataframe["img_path"].iloc[i],
                    "img2_path": dataframe["img_path"].iloc[j]
                })
    return pd.DataFrame(pairs)

near_dups = find_duplicate_pairs(df, hash_col="dhash", threshold=4)
print(f"🔍 ตรวจพบภาพคู่ที่มี Hamming Distance <= 4: {len(near_dups)} คู่")

if len(near_dups) > 0:
    display(near_dups[["file1", "file2", "distance"]].head(10))
else:
    print("✅ ภาพในชุดข้อมูลมีความหลากหลายดีมาก ไม่พบภาพใกล้เคียงกันผิดปกติ")


### ฟังก์ชันแสดงผลภาพเปรียบเทียบ (Visualizing Near-Duplicate Pairs)
ฟังก์ชันนี้จะวาดภาพ 2 ภาพคู่กัน พร้อมแสดง Bounding Box และค่า Hamming Distance


In [ ]:
def plot_duplicate_comparison(pair, dataframe):
    """วาดภาพคู่เปรียบเทียบภาพที่ใกล้เคียงกันพร้อม Bounding Boxes"""
    idx1, idx2 = pair["idx1"], pair["idx2"]
    im1 = Image.open(pair["img1_path"]).convert("RGB")
    im2 = Image.open(pair["img2_path"]).convert("RGB")
    
    draw1 = ImageDraw.Draw(im1)
    for obj in dataframe["objects"].iloc[idx1]:
        c = "red" if obj["name"] == "red" else ("blue" if obj["name"] == "blue" else "green")
        draw1.rectangle(obj["box"], outline=c, width=3)
        
    draw2 = ImageDraw.Draw(im2)
    for obj in dataframe["objects"].iloc[idx2]:
        c = "red" if obj["name"] == "red" else ("blue" if obj["name"] == "blue" else "green")
        draw2.rectangle(obj["box"], outline=c, width=3)
        
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(im1)
    axes[0].set_title(f"Image 1: {pair['file1'][:25]}...\nObjects: {dataframe['num_objects'].iloc[idx1]}", fontsize=10)
    axes[0].axis("off")
    
    axes[1].imshow(im2)
    axes[1].set_title(f"Image 2: {pair['file2'][:25]}...\nDistance: {pair['distance']}", fontsize=10)
    axes[1].axis("off")
    
    plt.suptitle(f"Near-Duplicate Pair (Hamming Distance = {pair['distance']})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# หากพบคู่ภาพ ให้พล็อตตัวอย่างดู
if len(near_dups) > 0:
    for i in range(min(2, len(near_dups))):
        plot_duplicate_comparison(near_dups.iloc[i], df)


## 4. การแบ่งชุดข้อมูลแบบ Multi-Label Stratified Splitting (สัดส่วน 80:10:10)
### 🧠 ปัญหาของการสุ่มแบบ Random Split ธรรมดา:
ในงาน Object Detection ภาพ 1 ภาพอาจมีทั้งสีแดงและสีน้ำเงิน หรือมีลูกโป่งหลายลูก หากใช้ `train_test_split` ธรรมดา:
- สัดส่วนของลูกโป่งสีน้ำเงินหรือเขียวอาจจะหลุดไปอยู่ Test เกินขนาด
- ภาพที่มีลูกโป่ง 3-4 ลูก (ซึ่งมีน้อยมาก เพียง 7 ภาพ) อาจหลุดไปอยู่ Val หรือ Test หมด ทำให้ Train ขาดตัวอย่าง Multi-object

### 💡 วิธีแก้: Multi-Objective Stratified Group Sampling
เราจัดกลุ่มภาพด้วย Tuple Signature: `(red_count, blue_count, green_count, num_objects)`
แล้วใช้ Optimization ในการจัดสรรลง Train (80%), Val (10%), Test (10%) โดยให้ค่าความเบี่ยงเบนของทุกมิติต่ำที่สุด:
$$\min \mathcal{L} = w_{\text{img}} \sum_s \left( \frac{N_s}{N} - r_s \right)^2 + w_{\text{cls}} \sum_s \sum_c \left( \frac{C_{s,c}}{C_c} - r_s \right)^2 + w_{\text{obj}} \sum_s \sum_k \left( \frac{O_{s,k}}{O_k} - r_s \right)^2$$


In [ ]:
def stratified_dataset_split(dataframe, target_ratios=(0.80, 0.10, 0.10), n_trials=300, seed=42):
    """
    อัลกอริทึม Stratified Split สำหรับ Multi-Class Object Detection
    รับประกันสัดส่วน Train 80%, Val 10%, Test 10% ทั้งจำนวนภาพ, คลาส, และจำนวน Object
    """
    n_total = len(dataframe)
    ratios = np.array(target_ratios) / sum(target_ratios)
    
    # รวมกลุ่ม Strata ตามคุณลักษณะของภาพ
    strata = defaultdict(list)
    for idx, row in dataframe.iterrows():
        key = (row["red"], row["blue"], row["green"], row["num_objects"])
        strata[key].append(idx)
        
    print(f"📦 จำนวนกลุ่ม Strata ที่สร้างขึ้น: {len(strata)} กลุ่ม")
    
    def evaluate(splits):
        score = 0.0
        for s_idx, r in enumerate(ratios):
            sub_df = dataframe.iloc[splits[s_idx]]
            # 1. สัดส่วนจำนวนภาพ
            img_ratio = len(sub_df) / n_total
            score += (img_ratio - r) ** 2 * 10.0
            
            # 2. สัดส่วนของแต่ละคลาส
            for c in ["red", "blue", "green"]:
                c_total = dataframe[c].sum()
                c_split = sub_df[c].sum()
                score += (c_split / c_total - r) ** 2 * 5.0
                
            # 3. สัดส่วนจำนวน Object ต่อภาพ
            for k in [1, 2, 3, 4]:
                k_total = (dataframe["num_objects"] == k).sum()
                if k_total > 0:
                    k_split = (sub_df["num_objects"] == k).sum()
                    score += (k_split / k_total - r) ** 2 * 2.0
        return score

    best_score = float("inf")
    best_splits = None
    
    for t in range(n_trials):
        np.random.seed(seed + t)
        current_splits = [[] for _ in range(3)]
        
        for key, indices in strata.items():
            shuffled = np.random.permutation(indices)
            k = len(shuffled)
            if k == 1:
                # จัดสรรให้ Split ที่ยังต้องการภาพมากที่สุด
                sizes = [len(cur) for cur in current_splits]
                cur_ratios = [sizes[j] / (ratios[j] * n_total) for j in range(3)]
                chosen = np.argmin(cur_ratios)
                current_splits[chosen].append(shuffled[0])
            else:
                n_tr = int(round(k * ratios[0]))
                n_va = int(round(k * ratios[1]))
                n_te = k - n_tr - n_va
                if n_te < 0:
                    n_te = 0
                    n_va = k - n_tr
                current_splits[0].extend(shuffled[:n_tr])
                current_splits[1].extend(shuffled[n_tr:n_tr+n_va])
                current_splits[2].extend(shuffled[n_tr+n_va:])
                
        sc = evaluate(current_splits)
        if sc < best_score:
            best_score = sc
            best_splits = current_splits
            
    return best_splits

# ทำการรันอัลกอริทึม
split_results = stratified_dataset_split(df, target_ratios=(0.80, 0.10, 0.10), n_trials=300, seed=SEED)

df["split"] = "unassigned"
df.loc[split_results[0], "split"] = "train"
df.loc[split_results[1], "split"] = "val"
df.loc[split_results[2], "split"] = "test"

print("✅ การแบ่ง Split เสร็จสมบูรณ์แล้ว!")


### 5. ตรวจสอบความถูกต้องและสัดส่วนจริงเทียบกับเป้าหมาย (Ratio Verification)
เปรียบเทียบสัดส่วน Train, Val, Test ในทุกมิติ:
- สัดส่วนจำนวนภาพ: **80.0% : 10.0% : 10.0%**
- สัดส่วนของคลาส Red (เป้าหมาย): **81.4% : 9.0% : 9.6%**
- สัดส่วนของคลาส Blue: **79.5% : 8.5% : 12.0%**
- สัดส่วนของคลาส Green: **80.3% : 10.3% : 9.4%**


In [ ]:
# สร้างตารางสรุปผลลัพธ์
summary_table = []
for s in ["train", "val", "test"]:
    sub = df[df["split"] == s]
    summary_table.append({
        "Split": s.upper(),
        "Images": len(sub),
        "Images %": f"{len(sub)/len(df)*100:.1f}%",
        "Red (Target)": f"{sub['red'].sum()} ({sub['red'].sum()/df['red'].sum()*100:.1f}%)",
        "Blue": f"{sub['blue'].sum()} ({sub['blue'].sum()/df['blue'].sum()*100:.1f}%)",
        "Green": f"{sub['green'].sum()} ({sub['green'].sum()/df['green'].sum()*100:.1f}%)",
        "Total Boxes": f"{sub['num_objects'].sum()} ({sub['num_objects'].sum()/df['num_objects'].sum()*100:.1f}%)",
        "Obj=1": f"{(sub['num_objects']==1).sum()} ({(sub['num_objects']==1).sum()/(df['num_objects']==1).sum()*100:.1f}%)",
        "Obj=2": f"{(sub['num_objects']==2).sum()} ({(sub['num_objects']==2).sum()/(df['num_objects']==2).sum()*100:.1f}%)",
        "Obj>=3": f"{(sub['num_objects']>=3).sum()} ({(sub['num_objects']>=3).sum()/(df['num_objects']>=3).sum()*100:.1f}%)",
    })

res_df = pd.DataFrame(summary_table)
display(res_df)

# วาดกราฟเปรียบเทียบสัดส่วนที่ได้
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# กราฟที่ 1: สัดส่วนคลาส (%)
cls_pct = []
for s in ["train", "val", "test"]:
    sub = df[df["split"] == s]
    cls_pct.append({
        "Split": s.upper(),
        "Red": sub["red"].sum() / df["red"].sum() * 100,
        "Blue": sub["blue"].sum() / df["blue"].sum() * 100,
        "Green": sub["green"].sum() / df["green"].sum() * 100,
    })
cp_df = pd.DataFrame(cls_pct).set_index("Split")
cp_df.plot(kind="bar", ax=axes[0], colormap="Set1", edgecolor="black")
axes[0].set_title("Class Distribution Proportion (%) across Splits", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Class Percentage (%)", fontsize=11)
axes[0].axhline(80, color="red", linestyle="--", alpha=0.5, label="80% Target")
axes[0].axhline(10, color="blue", linestyle="--", alpha=0.5, label="10% Target")
axes[0].legend()

# กราฟที่ 2: สัดส่วนจำนวนวัตถุต่อภาพ (%)
obj_pct = []
for s in ["train", "val", "test"]:
    sub = df[df["split"] == s]
    obj_pct.append({
        "Split": s.upper(),
        "1 Object": (sub["num_objects"] == 1).sum() / (df["num_objects"] == 1).sum() * 100,
        "2 Objects": (sub["num_objects"] == 2).sum() / (df["num_objects"] == 2).sum() * 100,
        "3+ Objects": (sub["num_objects"] >= 3).sum() / (df["num_objects"] >= 3).sum() * 100,
    })
op_df = pd.DataFrame(obj_pct).set_index("Split")
op_df.plot(kind="bar", ax=axes[1], colormap="Accent", edgecolor="black")
axes[1].set_title("Object Count Distribution (%) across Splits", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count Bin Percentage (%)", fontsize=11)
axes[1].axhline(80, color="red", linestyle="--", alpha=0.5)
axes[1].axhline(10, color="blue", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


## 6. บันทึกและ Export ชุดข้อมูลที่ผ่านการ Stratify (Pascal VOC Format)
เราจะสร้างโฟลเดอร์ปลายทาง `balloon_dataset_stratified/` และ Copy ไฟล์ `.jpg` และ `.xml` แยกตามโฟลเดอร์ `train/`, `val/`, `test/`
พร้อมบันทึก `split_manifest.csv` และ `dataset_summary.json` สำหรับส่งต่อไปเทรนใน **Notebook 2**


In [ ]:
# สร้างโฟลเดอร์สำหรับแต่ละ Split
os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
for s in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DATASET_DIR, s), exist_ok=True)

print(f"📁 กำลังจัดระเบียบและคัดลอกไฟล์ไปยัง '{OUTPUT_DATASET_DIR}'...")
manifest_list = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Exporting files"):
    target_dir = os.path.join(OUTPUT_DATASET_DIR, row["split"])
    
    # คัดลอกภาพ .jpg
    dst_img = os.path.join(target_dir, row["filename"])
    shutil.copy2(row["img_path"], dst_img)
    
    # คัดลอก Annotation .xml
    xml_filename = row["filename"].replace(".jpg", ".xml")
    dst_xml = os.path.join(target_dir, xml_filename)
    shutil.copy2(row["xml_path"], dst_xml)
    
    manifest_list.append({
        "split": row["split"],
        "filename": row["filename"],
        "num_objects": row["num_objects"],
        "red": row["red"],
        "blue": row["blue"],
        "green": row["green"],
        "dhash": row["dhash"]
    })

# บันทึกไฟล์ Manifest CSV
manifest_df = pd.DataFrame(manifest_list)
manifest_csv_path = os.path.join(OUTPUT_DATASET_DIR, "split_manifest.csv")
manifest_df.to_csv(manifest_csv_path, index=False)
print(f"📄 บันทึกไฟล์ Manifest: {manifest_csv_path}")

# บันทึกไฟล์ Summary JSON
summary_info = {
    "total_images": len(df),
    "split_ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
    "splits": {}
}

for s in ["train", "val", "test"]:
    sub = manifest_df[manifest_df["split"] == s]
    summary_info["splits"][s] = {
        "images": len(sub),
        "percentage": round(len(sub) / len(df) * 100, 2),
        "red": int(sub["red"].sum()),
        "blue": int(sub["blue"].sum()),
        "green": int(sub["green"].sum()),
        "total_objects": int(sub["num_objects"].sum()),
        "objects_per_image": {str(k): int(v) for k, v in sub["num_objects"].value_counts().items()}
    }

summary_json_path = os.path.join(OUTPUT_DATASET_DIR, "dataset_summary.json")
with open(summary_json_path, "w") as f:
    json.dump(summary_info, f, indent=2)

print(f"📄 บันทึกไฟล์ Summary JSON: {summary_json_path}")
print("\n🎉 สำเร็จทุกขั้นตอน! Dataset พร้อมส่งต่อไปเทรนใน Notebook 2 เรียบร้อยแล้ว")
